### Configuración de conexión

In [1]:
# Define las credenciales y las cadenas JDBC de la base fuente (solo lectura) y de la base destino (bodega)
db_user = 'DB_202615_ma_losada'
db_psswd = ''

db_host = '157.253.236.120'
db_port = 8080
db_fuente = 'RaSaTransaccional_ETL'      # base con las tablas Fuente*_ETL
db_destino = 'DB_202615_ma_losada'      # base donde se crean las dimensiones

if not db_user or not db_psswd or '<' in db_fuente or '<' in db_destino:
    raise ValueError('Complete db_user, db_psswd, db_fuente y db_destino antes de continuar')

jdbc_opciones = 'useConfigs=maxPerformance&connectTimeout=10000&socketTimeout=600000'

source_db_connection_string = f'jdbc:mysql://{db_host}:{db_port}/{db_fuente}?{jdbc_opciones}'
dest_db_connection_string = f'jdbc:mysql://{db_host}:{db_port}/{db_destino}?rewriteBatchedStatements=true&{jdbc_opciones}'

### Entorno e imports

In [2]:
# Detecta el entorno, importa las librerías y obtiene el conector JDBC de MySQL
import os
import sys
from pathlib import Path
import urllib.request

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_CODESPACES = os.environ.get('CODESPACES') == 'true'

if IN_COLAB or IN_CODESPACES:
    %pip install -q pyspark==4.2.0 pandas==2.3.3 numpy==2.5.3 pyarrow==25.0.1 matplotlib==3.11.2

if IN_COLAB:
    !apt-get install -y -qq openjdk-21-jdk-headless > /dev/null
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'

ENVIRONMENT = 'colab' if IN_COLAB else 'codespaces' if IN_CODESPACES else 'local'
print(f'Entorno detectado: {ENVIRONMENT}')

from pyspark.sql import functions as f, SparkSession, types as t
from pyspark.sql.functions import col
from pyspark.sql.window import Window

CONNECTOR_VERSION = '9.7.0'
CONNECTOR_URL = ('https://repo1.maven.org/maven2/com/mysql/mysql-connector-j/'
                 f'{CONNECTOR_VERSION}/mysql-connector-j-{CONNECTOR_VERSION}.jar')

def obtener_jar_mysql():
    env_path = os.environ.get('CONNECTOR_J_PATH', '')
    if env_path and Path(env_path).is_file():
        return Path(env_path)
    jar = Path.home() / '.miid4103' / 'jars' / f'mysql-connector-j-{CONNECTOR_VERSION}.jar'
    if not jar.exists():
        jar.parent.mkdir(parents=True, exist_ok=True)
        print(f'Descargando MySQL Connector/J {CONNECTOR_VERSION}...')
        parcial = jar.with_suffix('.part')
        urllib.request.urlretrieve(CONNECTOR_URL, parcial)
        parcial.replace(jar)
    return jar

MYSQL_JAR = obtener_jar_mysql()
MYSQL_JAR_URI = MYSQL_JAR.as_uri()
print(f'Conector JDBC: {MYSQL_JAR}')

Entorno detectado: local
Conector JDBC: C:\Users\ma.losada\.miid4103\jars\mysql-connector-j-9.7.0.jar


### Sesión de Spark

In [3]:
# Configura Python/Hadoop para Spark y crea la sesión local con el conector MySQL
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
import pyspark
os.environ.setdefault('SPARK_HOME', os.path.dirname(pyspark.__file__))
if os.name == 'nt' and 'HADOOP_HOME' not in os.environ:
    hadoop_home = Path.home() / '.miid4103' / 'hadoop'
    (hadoop_home / 'bin').mkdir(parents=True, exist_ok=True)
    (hadoop_home / 'bin' / 'winutils.exe').touch()
    os.environ['HADOOP_HOME'] = str(hadoop_home)

spark = (SparkSession.builder
         .master('local[*]')
         .appName('ETL-RaSA')
         .config('spark.driver.host', '127.0.0.1')
         .config('spark.jars', MYSQL_JAR_URI)
         .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.ui.enabled', 'false')
         .config('spark.ui.showConsoleProgress', 'false')
         .config('spark.sql.ansi.enabled', 'false')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

### Funciones de conexión

In [4]:
# Funciones para leer una consulta desde la base de datos y para guardar un dataframe en una tabla
def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    df_bd = spark.read.format('jdbc')\
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()
    return df_bd.cache()

def guardar_db(db_connection_string, df, tabla, db_user, db_psswd):
    df.select('*').write.format('jdbc') \
      .mode('append') \
      .option('url', db_connection_string) \
      .option('dbtable', tabla) \
      .option('user', db_user) \
      .option('password', db_psswd) \
      .option('driver', 'com.mysql.cj.jdbc.Driver') \
      .option('batchsize', 10000) \
      .option('numPartitions', 4) \
      .save()

### Extracción y limpieza de la fuente

In [5]:
# Lee la fuente sin duplicados y corrige áreas negativas y el error '0001' en la población
sql_fuente = f'(SELECT DISTINCT * FROM {db_fuente}.FuenteAreasDeServicio_ETL) AS Temp_areas'
fuente = obtener_dataframe_de_bd(source_db_connection_string, sql_fuente, db_user, db_psswd)

fuente = (fuente
          .withColumn('Area', f.abs(col('Area')))
          .withColumn('PoblacionAct', f.trim(col('PoblacionAct')))
          .withColumn('PoblacionAct', f.when(col('PoblacionAct').rlike('.+0001$'),
                                             f.regexp_replace('PoblacionAct', '0001$', ''))
                                       .otherwise(col('PoblacionAct')).cast('long'))
          .dropDuplicates())
fuente.show(5)

+------------------+--------------------+-------------+-----------------+----------+------------+-----+--------+-----+
|IdAreaDeServicio_T|NombreAreaDeServicio|IdGeografia_T|          Condado|    Estado|PoblacionAct| Area|Densidad|Fecha|
+------------------+--------------------+-------------+-----------------+----------+------------+-----+--------+-----+
|         100622017|New Jersey - Medi...|        34005|Burlington County|New Jersey|      464269|805.0|   577.0| 2017|
|         101152020|New Jersey - Medi...|        34035|  Somerset County|New Jersey|      345647|305.0|  1133.0| 2020|
|         101242019|New Jersey  - Med...|        34005|Burlington County|New Jersey|      464269|805.0|   577.0| 2019|
|         101852019|New Jersey  - Med...|        34017|    Hudson County|New Jersey|      702463| 47.0| 14946.0| 2019|
|          10272017|BlueOptions16842F...|        12059|    Holmes County|   Florida|       19784|482.0|    41.0| 2017|
+------------------+--------------------+-------

### Dimensión Geografia

In [6]:
# Deja una fila por geografía (la más reciente), le asigna el id de la bodega y la carga
w = Window.partitionBy('IdGeografia_T').orderBy(col('Fecha').desc())
geografia = (fuente.withColumn('rn', f.row_number().over(w)).filter('rn = 1')
             .selectExpr('IdGeografia_T', 'Estado', 'Condado', 'Area as AreaAct', 'Densidad as DensidadAct', 'PoblacionAct'))
geografia = geografia.coalesce(1).withColumn('IdGeografia_DWH', f.monotonically_increasing_id() + 1)
geografia = geografia.select('IdGeografia_DWH', 'IdGeografia_T', 'Estado', 'Condado', 'AreaAct', 'DensidadAct', 'PoblacionAct')
geografia.show(5)

guardar_db(dest_db_connection_string, geografia, f'{db_destino}.Geografia', db_user, db_psswd)

+---------------+-------------+-------+--------------+-------+-----------+------------+
|IdGeografia_DWH|IdGeografia_T| Estado|       Condado|AreaAct|DensidadAct|PoblacionAct|
+---------------+-------------+-------+--------------+-------+-----------+------------+
|              1|         1001|Alabama|Autauga County|  594.0|       99.0|       59095|
|              2|         1003|Alabama|Baldwin County| 1589.0|      151.0|      239294|
|              3|         1005|Alabama|Barbour County|  884.0|       28.0|       24964|
|              4|         1007|Alabama|   Bibb County|  622.0|       36.0|       22477|
|              5|         1009|Alabama| Blount County|  644.0|       92.0|       59041|
+---------------+-------------+-------+--------------+-------+-----------+------------+
only showing top 5 rows


### Dimensión AreasDeServicio

In [7]:
# Deja una fila por área de servicio con su nombre y año de creación, le asigna el id de la bodega y la carga
areas = (fuente.groupBy('IdAreaDeServicio_T')
         .agg(f.min('NombreAreaDeServicio').alias('Nombre'), f.min('Fecha').alias('AnnioCreacion')))
areas = areas.coalesce(1).withColumn('IdAreaDeServicio_DWH', f.monotonically_increasing_id() + 1)
areas = areas.select('IdAreaDeServicio_DWH', 'IdAreaDeServicio_T', 'Nombre', 'AnnioCreacion')
areas.show(5)

guardar_db(dest_db_connection_string, areas, f'{db_destino}.AreasDeServicio', db_user, db_psswd)

+--------------------+------------------+--------------------+-------------+
|IdAreaDeServicio_DWH|IdAreaDeServicio_T|              Nombre|AnnioCreacion|
+--------------------+------------------+--------------------+-------------+
|                   1|             12018|NoVa10207VA038000...|         2018|
|                   2|             22019|NoVA10207VA038000...|         2019|
|                   3|             32019|NoVA10207VA038000...|         2019|
|                   4|             42018|NoVa10207VA038000...|         2018|
|                   5|             52018|NoVa10207VA038000...|         2018|
+--------------------+------------------+--------------------+-------------+
only showing top 5 rows


### AsociacionAreaServicioGeografia

In [8]:
# Relaciona cada área con sus geografías usando los ids de la bodega y carga la tabla puente
asociacion = (fuente.select('IdAreaDeServicio_T', 'IdGeografia_T').distinct()
              .join(areas.select('IdAreaDeServicio_T', 'IdAreaDeServicio_DWH'), on='IdAreaDeServicio_T', how='inner')
              .join(geografia.select('IdGeografia_T', 'IdGeografia_DWH'), on='IdGeografia_T', how='inner')
              .select('IdAreaDeServicio_DWH', 'IdGeografia_DWH'))
asociacion.show(5)

guardar_db(dest_db_connection_string, asociacion, f'{db_destino}.AsociacionAreaServicioGeografia', db_user, db_psswd)

+--------------------+---------------+
|IdAreaDeServicio_DWH|IdGeografia_DWH|
+--------------------+---------------+
|               10203|            208|
|                   4|            220|
|                5839|            702|
|               10207|            628|
|               10209|            685|
+--------------------+---------------+
only showing top 5 rows
